# Experiment 2: Logit Lens Analysis at Position t-1

**Research Question:** At which transformer layer (1..32) does the divergence
between hallucinated and real objects first emerge at position t-1 (m=1)?

**Prerequisites:**
- Add the output of the Experiment 1 notebook as an input to this notebook
  (Kaggle: Add Data -> Notebook Output Files -> select Exp 1 notebook).
- Add the `biminhco/val2014` dataset as an input.

**This notebook does NOT re-run any part of Experiment 1.**

In [ ]:
# CELL 1: Environment setup and GPU check
import subprocess, os, sys

# GPU check
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

# Install dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.37.0", "accelerate>=0.26.0",
     "bitsandbytes", "sentencepiece", "tqdm", "scipy", "scikit-learn"],
    check=True
)
print("Dependencies installed.")

In [ ]:
# CELL 2: Clone repository and prepare workspace
import subprocess, os

REPO = "/kaggle/working/Experiment_CTI"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/ntmy12/Experiment_CTI.git", REPO],
        check=True
    )
else:
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO, check=True)

print("Repository ready at", REPO)
print("Files in src/:")
print(os.listdir(os.path.join(REPO, "src")))

In [ ]:
# CELL 3: Locate labels.jsonl from Experiment 1 notebook output
# Kaggle mounts the previous notebook output under /kaggle/input/<notebook-slug>/
# Adjust the path below to match your actual notebook slug.
import os, shutil, glob

REPO = "/kaggle/working/Experiment_CTI"

# Auto-search for labels.jsonl in all Kaggle input paths
candidates = glob.glob("/kaggle/input/**/labels.jsonl", recursive=True)
print("Found labels.jsonl candidates:", candidates)

if len(candidates) == 0:
    raise FileNotFoundError(
        "labels.jsonl not found in /kaggle/input/.\n"
        "Add the Experiment 1 notebook output as a data source for this notebook.\n"
        "Kaggle: Add Data -> Notebook Output Files -> select Exp 1 notebook."
    )

labels_src = candidates[0]
labels_dst = os.path.join(REPO, "data", "labels.jsonl")
os.makedirs(os.path.join(REPO, "data"), exist_ok=True)
shutil.copy(labels_src, labels_dst)

n_lines = sum(1 for _ in open(labels_dst))
print(f"Copied labels.jsonl ({n_lines} records) to {labels_dst}")

# Also locate synonyms.txt from repo (already cloned)
synonyms_path = os.path.join(REPO, "data", "synonyms.txt")
assert os.path.exists(synonyms_path), f"synonyms.txt not found at {synonyms_path}"
print("synonyms.txt found.")

In [ ]:
# CELL 4: Locate COCO val2014 images
import glob, os

# Find image directory from mounted dataset
sample = glob.glob("/kaggle/input/**/COCO_val2014_*.jpg", recursive=True)
if not sample:
    raise FileNotFoundError(
        "COCO val2014 images not found. Add biminhco/val2014 as a data source."
    )

IMAGE_DIR = os.path.dirname(sample[0])
total_imgs = len(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))
print(f"COCO val2014 images found at: {IMAGE_DIR}")
print(f"Total images: {total_imgs}")

In [ ]:
# CELL 5: Run Experiment 2 -- Logit Lens
import subprocess, sys, os

REPO = "/kaggle/working/Experiment_CTI"

result = subprocess.run(
    [
        sys.executable, "src/06_logit_lens.py",
        "--labels_file",   "data/labels.jsonl",
        "--synonyms_file", "data/synonyms.txt",
        "--image_dir",     IMAGE_DIR,
        "--model_name",    "llava-hf/llava-1.5-7b-hf",
        "--output_dir",    "results/exp2_logitlens",
        "--K",             "10",
        "--caliper",       "0.1",
        "--seed",          "0",
    ],
    cwd=REPO,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(f"Experiment 2 failed with exit code {result.returncode}.")
print("Experiment 2 (Logit Lens) completed.")

In [ ]:
# CELL 6: Inspect results
import pandas as pd, os

REPO = "/kaggle/working/Experiment_CTI"
df = pd.read_csv(os.path.join(REPO, "results", "exp2_logitlens", "layer_summary.csv"))

sig_df = df[df["holm_p"] < 0.05].sort_values("layer")
peak_row = df.loc[df["mean_delta"].idxmin()]

print("=" * 60)
print("EXPERIMENT 2 -- LOGIT LENS RESULTS")
print("=" * 60)
print(f"Layers Holm p < 0.05 : {sorted(sig_df['layer'].tolist())}")
if len(sig_df) > 0:
    print(f"First significant layer: Layer {int(sig_df['layer'].min())}")
print(f"Peak divergence layer: Layer {int(peak_row['layer'])} "
      f"(Delta = {float(peak_row['mean_delta']):.4f}, "
      f"Holm p = {float(peak_row['holm_p']):.4e})")

cols = ["layer", "mean_S_halluc", "mean_S_real", "mean_delta",
        "delta_ci_lo", "delta_ci_hi", "dz", "holm_p",
        "median_rank_halluc", "median_rank_real"]
print()
print(df[cols].to_string(index=False))

In [ ]:
# CELL 7: Display figures
from IPython.display import Image as IPyImage, display
import os

REPO = "/kaggle/working/Experiment_CTI"
fig_dir = os.path.join(REPO, "results", "exp2_logitlens", "figures")

for fig_name in ["s_vs_layer.png", "delta_vs_layer.png", "rank_vs_layer.png"]:
    path = os.path.join(fig_dir, fig_name)
    if os.path.exists(path):
        print(f"--- {fig_name} ---")
        display(IPyImage(filename=path))
    else:
        print(f"Not found: {fig_name}")

In [ ]:
# CELL 8: Package results for download
import subprocess, os

REPO = "/kaggle/working/Experiment_CTI"
archive = "/kaggle/working/exp2_logitlens_results.tar.gz"

subprocess.run(
    ["tar", "-czf", archive,
     "-C", os.path.join(REPO, "results"),
     "exp2_logitlens"],
    check=True
)

size_mb = os.path.getsize(archive) / 1024 / 1024
print(f"Archive: {archive} ({size_mb:.1f} MB)")
print("Download from Kaggle Output panel.")